# 1.2 • Candidates are not matches

**Learning objectives:** build an R-tree; refine bounding-box candidates with exact predicates; measure indexing costs.

**How to learn:** predict - run - explain - change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

An R-tree indexes rectangles. A point may hit a polygon bounding box while falling in a hole. Nearest bounding box also does not establish watershed membership. We use `covers` so shared boundaries may legitimately return multiple polygons.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Build a counterexample


In [2]:
from shapely.geometry import Polygon, Point, box
from rtree import index
from time import perf_counter
polygons=[Polygon([(0,0),(3,0),(3,3),(0,3)], holes=[[(1,1),(2,1),(2,2),(1,2)]]),box(3,0,4,3)]
idx=index.Index((i,g.bounds,None) for i,g in enumerate(polygons))
p=Point(1.5,1.5)
candidates=list(idx.intersection(p.bounds))
exact=[i for i in candidates if polygons[i].covers(p)]
assert candidates == [0] and exact == []
print('Bounding-box candidates:',candidates,'Exact matches:',exact)


Bounding-box candidates: [0] Exact matches: []


## Compare identical answers and separate build time


In [3]:
polygons=[box(i,j,i+.9,j+.9) for i in range(20) for j in range(20)]
rng=np.random.default_rng(42)
queries=[Point(x,y) for x,y in rng.uniform(0,20,(300,2))]
t0=perf_counter(); idx=index.Index((i,g.bounds,None) for i,g in enumerate(polygons)); build=perf_counter()-t0
t0=perf_counter(); brute=[[i for i,g in enumerate(polygons) if g.covers(p)] for p in queries]; slow=perf_counter()-t0
t0=perf_counter(); fast=[[i for i in idx.intersection(p.bounds) if polygons[i].covers(p)] for p in queries]; indexed=perf_counter()-t0
assert [sorted(x) for x in fast] == brute
print(dict(polygons=len(polygons),queries=len(queries),build_seconds=build,scan_seconds=slow,index_seconds=indexed))


{'polygons': 400, 'queries': 300, 'build_seconds': 0.004575999977532774, 'scan_seconds': 0.6473424999858253, 'index_seconds': 0.011829399969428778}


## Your turn

Add `Point(3,1)` to the first example. Compare `covers` and `contains`. Repeat the timing with only four polygons.

Use a new cell below to try this before opening the solution.


In [4]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning - open after trying</summary>

Both adjacent polygons cover the shared edge; neither contains that boundary point in its interior. For tiny data, constructing the index may cost more than scanning. Report build plus query time, not just the fastest query.

</details>


## Check yourself

Expected: the hole produces one candidate and zero matches; indexed and brute-force results agree exactly. Why is a single winner at a shared boundary a policy decision?

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

Load projected watershed polygons and transform station points into their CRS first. Check invalid geometries explicitly. Preserve unmatched and multiple-match rows instead of silently assigning a nearest watershed.
